# AMPS I · Week 16 — Rendering the final report on Colab (and the optional adjoint extension)

1. **Render `report.qmd` → `report.html`** with Quarto (about 1 minute for the download, once per runtime). Skip this if you have Quarto on your own computer:
   `quarto render report.qmd` in the report folder.
2. **Optional:** install `dolfinx-adjoint` into the `%%fenicsx` environment for the bonus inverse-problem section (W16_final_project, §5).

The report itself contains no code cells: all numbers and figures come from your `project.ipynb`.

## 1 · Your report folder
Put `report.qmd`, `references.bib` and `figs/` in a folder on Google Drive (e.g. `MyDrive/AMPS1_project`) and set the path below.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
REPORT_DIR = "/content/drive/MyDrive/AMPS1_project"      # <- change if needed
print(sorted(os.listdir(REPORT_DIR)))

## 2 · Install Quarto (once per runtime) and render

In [ ]:
import os, subprocess
V = "1.9.36"
QUARTO = f"/content/quarto-{V}/bin/quarto"
if not os.path.exists(QUARTO):
    url = f"https://github.com/quarto-dev/quarto-cli/releases/download/v{V}/quarto-{V}-linux-amd64.tar.gz"
    subprocess.run(["wget", "-q", "-O", "/content/quarto.tar.gz", url], check=True)
    subprocess.run(["tar", "-xzf", "/content/quarto.tar.gz", "-C", "/content"], check=True)
print(subprocess.run([QUARTO, "--version"], capture_output=True, text=True).stdout.strip())

In [ ]:
r = subprocess.run([QUARTO, "render", "report.qmd", "--to", "html"], cwd=REPORT_DIR, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-3000:])
print("report.html written:", os.path.exists(os.path.join(REPORT_DIR, "report.html")))

Open `report.html` from Drive (download it) and check that the figures appear and the equations render. With `embed-resources: true` (in the template) the
HTML file is self-contained — that single file is what you submit together with `report.qmd`.

---
## 3 · Optional: dolfinx-adjoint for the inverse-problem bonus
Run the DOLFINx setup cell first (as in every weekly notebook), then the install cell. `--no-deps` keeps DOLFINx 0.11 untouched.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

In [ ]:
import subprocess
PY = "/content/micromamba/envs/fenicsx/bin/python"
MM = "/content/micromamba/bin/micromamba"

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr)[-1500:]

def has(mod):
    return run([PY, "-c", f"import {mod}"])[0]

if not has("pip"):                                  # the %%fenicsx environment ships without pip
    ok, log = run([PY, "-m", "ensurepip", "--upgrade"])
    print("ensurepip:", "ok" if ok else "failed")
    if not ok:
        ok, log = run([MM, "install", "-y", "-n", "fenicsx", "-c", "conda-forge", "--freeze-installed", "pip"])
        print("micromamba install pip:", "ok" if ok else "FAILED\n" + log)

def pip(*pkgs):
    ok, log = run([PY, "-m", "pip", "install", "--no-deps", "-q", *pkgs])
    print("pip install", *pkgs, "->", "ok" if ok else "FAILED\n" + log)

pip("dolfinx-adjoint==0.4.1", "pyadjoint-ad", "checkpoint_schedules", "fenicsx-compat")
for mod in ("packaging", "scipy", "mpmath", "sympy", "h5py"):
    if not has(mod):
        pip(mod)

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import pyadjoint
import dolfinx_adjoint as da
if MPI.COMM_WORLD.rank == 0:
    print("pyadjoint", getattr(pyadjoint, "__version__", "?"), "| dolfinx_adjoint", da.__file__)

Documentation and examples: <https://scientificcomputing.github.io/dolfinx-adjoint/README.html>. Use `da.Function`, `da.Constant`, `da.NonlinearProblem`
(or `da.LinearProblem`) and `da.assemble_scalar` so that the forward solve is recorded; `pyadjoint.Control`, `pyadjoint.ReducedFunctional`,
`pyadjoint.taylor_test` and `pyadjoint.minimize` do the rest. The three checks required for the bonus are listed in `W16_final_project.html` §5.